# Model Training and Evaluation
You should build a machine learning pipeline with a complete model training and evaluation step. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Conduct data exploration, data preprocessing, and feature engineering if necessary.
- Choose a few machine learning algorithms, such as [KNN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html), [decision tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html), and [gradient boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html).
- Define a grid of hyperparameters for every selected model.
- Conduct [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) or [random search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) using k-fold cross-validation on the training set to find out the best model (i.e., the best algorithm and its hyperparameters).
- Train the best model on the whole training set.
- Test the trained model on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods. Use them in practice.

Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [5]:

# Use the raw file URL
url = "https://raw.githubusercontent.com/m-mahdavi/teaching/main/datasets/mnist.csv"
df = pd.read_csv(url)

# Show the first 10 rows
df.head(10)

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,61207,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,33799,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,5414,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,61377,6,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1875,9,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# Basic EDA
print("\nMissing values:")
print(df.isnull().sum().sum())



Missing values:
0


In [11]:
#Train/Test Split
X=df.drop(['id','class'],axis=1)
y=df['class']

In [12]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Define Models & Pipelines


In [19]:
# PCA to reduce dimensionality
#pca
from sklearn.decomposition import PCA
pca = PCA(n_components=50, random_state=42)

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", pca),
    ("classifier", KNeighborsClassifier())
])

knn_param_grid = {
    "classifier__n_neighbors": [3, 5],
    "classifier__weights": ["uniform"],
    "classifier__metric": ["euclidean"]
}

dt_pipeline = Pipeline([
    ("pca", pca),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

dt_param_grid = {
    "classifier__max_depth": [None, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__criterion": ["gini"]
}

gb_pipeline = Pipeline([
    ("pca", pca),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

gb_param_grid = {
    "classifier__n_estimators": [100],
    "classifier__learning_rate": [0.1],
    "classifier__max_depth": [3]
}


Hyperparameter Tuning

In [20]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

models = {
    "KNN": (knn_pipeline, knn_param_grid),
    "Decision Tree": (dt_pipeline, dt_param_grid),
    "Gradient Boosting": (gb_pipeline, gb_param_grid)
}

best_models = {}

for name, (pipeline, param_grid) in models.items():
    print(f"\nRunning GridSearch for {name}...")
    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)
    best_models[name] = grid

    print("Best Parameters:", grid.best_params_)
    print("Best CV Score:", grid.best_score_)



Running GridSearch for KNN...
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best Parameters: {'classifier__metric': 'euclidean', 'classifier__n_neighbors': 5, 'classifier__weights': 'uniform'}
Best CV Score: 0.8965637497193947

Running GridSearch for Decision Tree...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best Parameters: {'classifier__criterion': 'gini', 'classifier__max_depth': None, 'classifier__min_samples_split': 2}
Best CV Score: 0.7256283654322377

Running GridSearch for Gradient Boosting...
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best Parameters: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 100}
Best CV Score: 0.8581288797532197


 Select Best Model

In [21]:
best_model_name = max(best_models, key=lambda x: best_models[x].best_score_)
best_model = best_models[best_model_name]

print(f"\nBest Model Overall: {best_model_name}")
print("Best Hyperparameters:", best_model.best_params_)


Best Model Overall: KNN
Best Hyperparameters: {'classifier__metric': 'euclidean', 'classifier__n_neighbors': 5, 'classifier__weights': 'uniform'}


Train Best Model on Full Training Set

In [22]:
best_model.fit(X_train, y_train)

Fitting 3 folds for each of 2 candidates, totalling 6 fits


GridSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('pca',
                                        PCA(n_components=50, random_state=42)),
                                       ('classifier', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'classifier__metric': ['euclidean'],
                         'classifier__n_neighbors': [3, 5],
                         'classifier__weights': ['uniform']},
             scoring='accuracy', verbose=1)

Evaluation on Test Set

In [23]:
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print("\n===== TEST SET EVALUATION =====")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))



===== TEST SET EVALUATION =====
Accuracy:  0.9175
Precision: 0.9184
Recall:    0.9175
F1 Score:  0.9170

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95        75
           1       0.95      0.99      0.97        97
           2       0.91      0.95      0.93        78
           3       0.92      0.92      0.92        84
           4       0.83      0.91      0.86        74
           5       0.90      0.86      0.88        73
           6       0.95      0.97      0.96        78
           7       0.93      0.94      0.94        85
           8       0.97      0.83      0.90        83
           9       0.87      0.81      0.84        73

    accuracy                           0.92       800
   macro avg       0.92      0.92      0.91       800
weighted avg       0.92      0.92      0.92       800

Confusion Matrix:
[[73  0  0  0  0  1  1  0  0  0]
 [ 0 96  1  0  0  0  0  0  0  0]
 [ 0  0 74  1  1  0  1  1  0